In [ ]:
import xarray as xr

from conus_biomass import dir_info

In [ ]:
dir_input = dir_info.dir_data_on_ref_grid + "1000m/"

In [ ]:
ds = xr.open_zarr(dir_input + "all_variables_2D.zarr")

In [ ]:
target_path = dir_input + "all_variables.nc"
print(target_path)

In [ ]:
chunk_size = 1000
for var in ds.data_vars:
    print(var)
    if var in ["x", "y"]:
        # print(var)
        ds[var] = ds[var].chunk({var: chunk_size})
    elif var not in ["band", "spatial_ref", "year"]:
        # print(var)
        ds[var] = ds[var].chunk({"x": chunk_size, "y": chunk_size})

In [ ]:
ds = ds.squeeze()
ds = ds.drop_vars("spatial_ref")
ds = ds.chunk({"x": chunk_size, "y": chunk_size})

In [ ]:
from dask.diagnostics import ProgressBar

encoding = {}
for var in ds.data_vars:
    dims = ds[var].dims
    chunks = []
    for dim in dims:
        if dim == "time" or dim == "year":
            chunks.append(1)
        elif dim in ["x", "y"]:
            dim_size = ds.sizes[dim]
            chunks.append(min(1000, dim_size))
        else:
            chunks.append(ds.sizes[dim])

    encoding[var] = {"chunksizes": tuple(chunks), "zlib": True, "complevel": 1}

print(f"Writing {len(ds.data_vars)} variables to {target_path}", flush=True)
print(f"Estimated size: {ds.nbytes / 1e9:.2f} GB", flush=True)

import time

start = time.time()
with ProgressBar():
    ds.to_netcdf(target_path, encoding=encoding)
elapsed = time.time() - start
print(f"Complete in {elapsed:.2f}s ({ds.nbytes/1e9/elapsed:.2f} GB/s)", flush=True)

In [ ]:
ds_test = xr.open_dataset(target_path)

In [ ]:
ds_test